In [1]:
import os
import time
import numpy as np
import faiss
import pypdf
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai

# 1. Load and parse the FIDE Laws of Chess PDF
pdf_path = "LawsOfChess.pdf"
reader = pypdf.PdfReader(pdf_path)

corpus_chunks = []
chunk_id = 0
chunk_size = 750
overlap = 120
step = chunk_size - overlap  # 630

for page_idx, page in enumerate(reader.pages):
    raw_text = page.extract_text() or ""
    # Normalize whitespace and excessive newlines into coherent prose
    clean_text = " ".join(raw_text.split())
    if len(clean_text) < 40:
        continue

    for start in range(0, len(clean_text), step):
        chunk_str = clean_text[start : start + chunk_size]
        if len(chunk_str.strip()) > 40:
            corpus_chunks.append({
                "chunk_id": chunk_id,
                "page": page_idx + 1,
                "text": chunk_str
            })
            chunk_id += 1

print(f"Total corpus chunks generated: {len(corpus_chunks)}")

Total corpus chunks generated: 97


In [2]:
# 2. Dense Embedding Model & FAISS Vector Index
bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

chunk_texts = [c["text"] for c in corpus_chunks]
corpus_embeddings = bi_encoder.encode(
    chunk_texts, 
    batch_size=64, 
    show_progress_bar=True, 
    normalize_embeddings=True
)

embedding_dim = corpus_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)  # Inner Product on normalized vectors = Cosine Similarity
index.add(np.array(corpus_embeddings, dtype=np.float32))

print(f"FAISS index built successfully with {index.ntotal} vectors of dimension {embedding_dim}.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

FAISS index built successfully with 97 vectors of dimension 384.


In [3]:
import os
import time
from google import genai
from sentence_transformers import CrossEncoder

# 1. Initialize Cross-Encoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# 2. Define the Prompt Template
PROMPT_TEMPLATE = """You are a helpful assistant answering strictly from the provided context.
If the answer cannot be found in the context, say "I do not have enough information."

Context:
{context}

Question: {query}
Answer:"""

# 3. Setup Gemini API Client with Fallback
api_key = os.environ.get("GEMINI_API_KEY2")

if not api_key:
    raise ValueError("GEMINI_API_KEY2 environment variable is not set!")

client = genai.Client(api_key=api_key)

def call_llm(prompt: str, max_retries: int = 5, backoff: float = 2.0) -> str:
    global client, api_key
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemma-4-26b-a4b-it",  # Or "gemma-4-31b-it" / "gemini-2.0-flash" from your enabled list
                contents=prompt
            )
            return response.text.strip()
        except Exception as e:
            if attempt == max_retries - 1:
                raise e
            print(f"[Warning] API Error ({e}). Retrying in {backoff:.1f}s... (Attempt {attempt + 1}/{max_retries})")
            time.sleep(backoff)
            backoff *= 2
            client = genai.Client(api_key=api_key)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [4]:
# 4. Pipeline 1 (Baseline Bi-Encoder) & Pipeline 2 (Cross-Encoder Re-Ranking)

def run_pipeline_1(query: str, top_k_retrieve: int = 10, top_k_context: int = 3):
    start_total = time.perf_counter()
    
    # Bi-encoder search
    t0 = time.perf_counter()
    query_vector = bi_encoder.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(query_vector, dtype=np.float32), top_k_retrieve)
    retrieval_time = time.perf_counter() - t0
    
    top_10_candidates = []
    for rank, idx in enumerate(indices[0]):
        item = corpus_chunks[idx].copy()
        item["baseline_rank"] = rank + 1
        item["bi_score"] = float(scores[0][rank])
        top_10_candidates.append(item)
        
    # Baseline selects Top 3 directly
    selected_top_3 = top_10_candidates[:top_k_context]
    context_str = "\n\n---\n\n".join(
        [f"[Chunk ID {c['chunk_id']} | Page {c['page']}]: {c['text']}" for c in selected_top_3]
    )
    
    # LLM Inference
    t_llm = time.perf_counter()
    prompt = PROMPT_TEMPLATE.format(context=context_str, query=query)
    answer = call_llm(prompt)
    llm_time = time.perf_counter() - t_llm
    
    total_time = time.perf_counter() - start_total
    
    return {
        "answer": answer,
        "top_10_candidates": top_10_candidates,
        "selected_chunks": selected_top_3,
        "retrieval_time": retrieval_time,
        "llm_time": llm_time,
        "total_time": total_time
    }

def run_pipeline_2(query: str, top_10_candidates: list, top_k_context: int = 3):
    start_total = time.perf_counter()
    
    # Cross-encoder joint scoring
    pairs = [[query, c["text"]] for c in top_10_candidates]
    t0 = time.perf_counter()
    ce_scores = cross_encoder.predict(pairs)
    rerank_time = time.perf_counter() - t0
    
    reranked_candidates = []
    for chunk, score in zip(top_10_candidates, ce_scores):
        c_copy = chunk.copy()
        c_copy["cross_score"] = float(score)
        reranked_candidates.append(c_copy)
        
    # Sort descending by Cross-Encoder logit score
    reranked_candidates.sort(key=lambda x: x["cross_score"], reverse=True)
    for new_rank, c in enumerate(reranked_candidates):
        c["reranked_rank"] = new_rank + 1
        
    # Re-ranked pipeline selects new Top 3
    selected_top_3 = reranked_candidates[:top_k_context]
    context_str = "\n\n---\n\n".join(
        [f"[Chunk ID {c['chunk_id']} | Page {c['page']}]: {c['text']}" for c in selected_top_3]
    )
    
    # LLM Inference with identical prompt
    t_llm = time.perf_counter()
    prompt = PROMPT_TEMPLATE.format(context=context_str, query=query)
    answer = call_llm(prompt)
    llm_time = time.perf_counter() - t_llm
    
    total_time = time.perf_counter() - start_total
    
    return {
        "answer": answer,
        "reranked_chunks": reranked_candidates,
        "selected_chunks": selected_top_3,
        "rerank_time": rerank_time,
        "llm_time": llm_time,
        "total_time": total_time
    }

In [8]:
# 5. Stress Testing Evaluation
stress_queries = [
    # Failure Mode 1: Paraphrase & Exception Override (Electronic Device Draw Exception)
    # Chunk 50 leaps from Baseline Rank 5 -> Re-Ranked Rank 1 (+4)
    "Under what circumstance does an opponent receive half a point rather than a full victory when an electronic communication device penalty occurs?",

    # Failure Mode 2: Unsupervised Venue Procedure Override (Appendix D.1 vs Article 10.2)
    # Chunk 73 leaps from Baseline Rank 6 -> Re-Ranked Rank 1 (+5)
    "If no arbiter is present in the venue during a quickplay finish, what specific documentation must a player submit to claim a draw on the basis that their opponent has been making no effort to win by normal means?"
]

profiling_log = []

for q in stress_queries:
    print("=" * 80)
    print(f"QUERY: {q}\n")
    
    p1 = run_pipeline_1(q)
    p2 = run_pipeline_2(q, p1["top_10_candidates"])
    
    print("--- RANK SHIFTS (Top 10 Baseline -> Re-Ranked) ---")
    print(f"{'Chunk ID':<10} | {'Page':<6} | {'Baseline Rank':<15} | {'Re-Ranked Rank':<15} | {'Shift'}")
    print("-" * 65)
    for c in p2["reranked_chunks"]:
        shift = c["baseline_rank"] - c["reranked_rank"]
        shift_str = f"+{shift}" if shift > 0 else str(shift)
        print(f"{c['chunk_id']:<10} | {c['page']:<6} | {c['baseline_rank']:<15} | {c['reranked_rank']:<15} | {shift_str}")
        
    print("\n--- LLM ANSWER 1 (Baseline Top-3 Chunks) ---")
    print(p1["answer"])
    
    print("\n--- LLM ANSWER 2 (Cross-Encoder Top-3 Chunks) ---")
    print(p2["answer"])
    
    print("\n--- LATENCY BREAKDOWN ---")
    p2_total_e2e = p1["retrieval_time"] + p2["rerank_time"] + p2["llm_time"]
    print(f"Baseline Pipeline : Retrieval: {p1['retrieval_time']:.4f}s | LLM: {p1['llm_time']:.4f}s | Total: {p1['total_time']:.4f}s")
    print(f"Re-Ranking Pipeline: Re-rank:   {p2['rerank_time']:.4f}s | LLM: {p2['llm_time']:.4f}s | Total E2E: {p2_total_e2e:.4f}s")
    print("\n")
    
    profiling_log.append({
        "query": q,
        "p1_retrieval": p1["retrieval_time"],
        "p1_llm": p1["llm_time"],
        "p1_total": p1["total_time"],
        "p2_rerank": p2["rerank_time"],
        "p2_llm": p2["llm_time"],
        "p2_total": p2_total_e2e
    })

QUERY: Under what circumstance does an opponent receive half a point rather than a full victory when an electronic communication device penalty occurs?

--- RANK SHIFTS (Top 10 Baseline -> Re-Ranked) ---
Chunk ID   | Page   | Baseline Rank   | Re-Ranked Rank  | Shift
-----------------------------------------------------------------
50         | 14     | 5               | 1               | +4
63         | 17     | 1               | 2               | -1
52         | 14     | 4               | 3               | +1
49         | 14     | 2               | 4               | -2
54         | 15     | 8               | 5               | +3
56         | 15     | 3               | 6               | -3
55         | 15     | 6               | 7               | -1
73         | 20     | 7               | 8               | -1
53         | 14     | 9               | 9               | 0
46         | 13     | 10              | 10              | 0

--- LLM ANSWER 1 (Baseline Top-3 Chunks) ---
I do not hav

In [9]:
# 6. Tabulate Latency for Section 3 of the Report
print(f"{'Query Index':<12} | {'Baseline Retr.':<16} | {'Re-Rank Time':<14} | {'Pipe 1 Total':<14} | {'Pipe 2 Total'}")
print("-" * 75)
for i, item in enumerate(profiling_log, 1):
    print(f"Query {i:<7} | {item['p1_retrieval'] * 1000:>10.2f} ms     | {item['p2_rerank'] * 1000:>8.2f} ms   | {item['p1_total']:>10.3f} s    | {item['p2_total']:>10.3f} s")

Query Index  | Baseline Retr.   | Re-Rank Time   | Pipe 1 Total   | Pipe 2 Total
---------------------------------------------------------------------------
Query 1       |      22.34 ms     |    53.80 ms   |     11.319 s    |      8.814 s
Query 2       |       8.98 ms     |    55.97 ms   |     11.371 s    |      9.838 s
